# Test Registered WNV Assistant Model

Loads a Unity Catalog model version and verifies an analytics follow-up using durable conversation memory.

Before running this notebook, run `resources/sql/create_assistant_tables.sql` once and register a model version built from the conversation-memory branch.

In [ ]:
%pip install databricks-sql-connector mlflow
dbutils.library.restartPython()

In [ ]:
import os
import uuid

import mlflow
import mlflow.pyfunc

# Enter the registered model version and the SQL warehouse to use for this test.
dbutils.widgets.text("model_version", "")
dbutils.widgets.text("warehouse_id", "")
dbutils.widgets.text("llm_endpoint", "databricks-gpt-oss-20b")

model_version = dbutils.widgets.get("model_version").strip()
warehouse_id = dbutils.widgets.get("warehouse_id").strip()
if not model_version or not warehouse_id:
    raise ValueError("Enter both model_version and warehouse_id in the widgets.")

# These notebook-scoped credentials are used only for this test.
context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
os.environ["WNV_DATABRICKS_HOST"] = context.apiUrl().get()
os.environ["WNV_DATABRICKS_TOKEN"] = context.apiToken().get()
os.environ["WNV_DATABRICKS_WAREHOUSE_ID"] = warehouse_id
os.environ["WNV_LLM_ENDPOINT"] = dbutils.widgets.get("llm_endpoint").strip()
os.environ["WNV_CONVERSATION_STORE"] = "delta"

mlflow.set_registry_uri("databricks-uc")
registered_uri = f"models:/eliao.wnv_demo.wnv_assistant/{model_version}"
model = mlflow.pyfunc.load_model(registered_uri)

conversation_id = str(uuid.uuid4())
user_id = context.userName().get()
print(f"Testing {registered_uri} with conversation {conversation_id}")

In [ ]:
first_result = model.predict({
    "question": "Show top 5 counties by mosquito activity in 2022",
    "conversation_id": conversation_id,
    "user_id": user_id,
})
print(first_result)

In [ ]:
follow_up_result = model.predict({
    "question": "What about 2021?",
    "conversation_id": conversation_id,  # Keep this the same.
    "user_id": user_id,                  # Keep this the same.
})
print(follow_up_result)

# Verify the model remembered the earlier 2022 context and then applied 2021.
assert follow_up_result[0]["route"] == "ANALYTICS"
assert follow_up_result[0]["tool_results"][0]["success"]
print("Conversation-memory follow-up test passed.")